I want to save a figure with the plot of each feature against time (both for GFOC and SWMA) and the Residuals of the two.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

In [2]:
# --- Load DataFrame ---
# local_head = 'C:/Users/david/Documents/David/Unibe/Master_Thesis/'
local_head = '/home/dschwarz/Documents/MT/'

# data directory
GFOC_dir = local_head+'Dataset/Dataset_MSc/GFOC_RDCDFI.parquet'
SWMA_dir = local_head+'Dataset/Dataset_MSc/SWMA_RDAWFI.parquet'

# remove first n features because of RAM issues
# find all column names
pf = pq.ParquetFile(GFOC_dir)
all_cols = pf.schema.names  # list of all column names
# picking columns
n = 110
cols_to_use = all_cols[n:]

# Load Data from parquet files
GFOC_data = pd.read_parquet(GFOC_dir, columns=cols_to_use).reset_index()
SWMA_data = pd.read_parquet(SWMA_dir, columns=cols_to_use).reset_index()

FileNotFoundError: [WinError 3] Failed to open local file '/home/dschwarz/Documents/MT/Dataset/Dataset_MSc/GFOC_RDCDFI.parquet'. Detail: [Windows error 3] The system cannot find the path specified.


In [ ]:
import matplotlib.dates as mdates
import re

# =========================== Input ===================================
# Add an option for monthly, daily, or hourly ticks
tick_interval = 'monthly'  # Change to 'monthly', 'daily', or 'hourly'
tick_step = 2  # Step for the ticks (e.g., every month =1, every 2 months = 2, etc)
# =====================================================================

# Helper function for tick formatting
def format_ticks(ax, tick_interval, tick_step):
    if tick_interval == 'monthly':
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=tick_step))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%b'))
    elif tick_interval == 'daily':
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=tick_step))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    elif tick_interval == 'hourly':
        ax.xaxis.set_major_locator(mdates.HourLocator(interval=tick_step))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
    ax.tick_params(axis='x', rotation=60)

# Time conversion
GFOC_time = pd.to_datetime(
    GFOC_data['time'].to_numpy(), 
    format='%Y-%m-%d %H:%M:%S'
).copy()
SWMA_time = pd.to_datetime(
    SWMA_data['time'].to_numpy(),
    format='%Y-%m-%d %H:%M:%S'
).copy()


features = GFOC_data.columns.tolist()
# Remove 'time' from features if present
if 'time' in features:
    features.remove('time')

for i in range(len(features)):

    feature = features[i]
    # Check if the feature exists in both datasets 
    if feature not in GFOC_data.columns or feature not in SWMA_data.columns:
        print(f"Feature '{feature}' not found in both datasets.")
        continue
    
    #Residuals: GFOC - SWMA
    GFOC_feature = GFOC_data.loc[:, feature].astype(float).copy()
    SWMA_feature = SWMA_data.loc[:, feature].astype(float).copy()

    if GFOC_feature.isna().all() or SWMA_feature.isna().all():
        print(f"Feature '{feature}' is empty or all NaN. Skipping.")
        continue

    
    try:
        residuals = GFOC_feature - SWMA_feature
    except ValueError as e:
        print(f"Error calculating residuals for feature '{feature}': {e}")
        continue

    # Create a figure with three subplots
    fig, axs = plt.subplots(3, 1, figsize=(10, 12))

    # Plot |avg B| against time for GFOC_data
    axs[0].plot(GFOC_time, GFOC_feature, 'k', label='GFOC: ' + feature + ' vs Time')
    axs[0].set_title('GFOC: Plot of ' + feature + ' against Time')
    axs[0].set_ylabel(feature)
    axs[0].legend(loc='upper right')
    axs[0].grid()
    format_ticks(axs[0], tick_interval, tick_step)

    # Plot |avg B| against time for SWMA_data
    axs[1].plot(SWMA_time, SWMA_feature, 'b', label='SWMA: ' + feature + ' vs Time')
    axs[1].set_title('SWMA: Plot of ' + feature + ' against Time')
    axs[1].set_ylabel(feature)
    axs[1].legend(loc='upper right')
    axs[1].grid()
    format_ticks(axs[1], tick_interval, tick_step)

    # Plot residuals against time
    axs[2].plot(GFOC_time, residuals, 'r', label='Residuals: GFOC - SWMA')
    axs[2].set_title('Residuals: GFOC - SWMA of ' + feature + ' against Time')
    axs[2].set_ylabel(feature + ' Residuals')
    axs[2].legend(loc='upper right')
    axs[2].grid()
    format_ticks(axs[2], tick_interval, tick_step)

    # Set the figure title
    fig.suptitle('Analysis of ' + feature, fontsize=16)

    plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit the title
    safe_feature = re.sub(r'[\\/:"*?<>|]+', "_", feature)  # replace bad chars
    plt.savefig(f'/home/dschwarz/Documents/MT/Dataset/Dataset_MSc/Features/{str(i+1+n)}_{safe_feature}.png', dpi=300)
    plt.close(fig)  # Close the figure to avoid RuntimeWarning for too many open figures
    del fig, axs

Features 29:35, 40:48 were empty or all NaN:

rho [kg/m^3]\
air_R [nm/s^2]\
air_S [nm/s^2]\
air_W [nm/s^2]\
VX_EF [m/s]\
VY_EF [m/s]\
VZ_EF [m/s]

srp_R [nm/s^2]\
srp_S [nm/s^2]\
srp_W [nm/s^2]\
ref_R [nm/s^2]\
ref_S [nm/s^2]\
ref_W [nm/s^2]\
emi_R [nm/s^2]\
emi_S [nm/s^2]\
emi_W [nm/s^2]

